# Step 5 Angle-wise Runner

This notebook runs Step 5 in chunks, one angle per cell, and writes logs/plots to separate folders.

Angles: `0, -60, 90, 120, -150`

Each run writes to: `src/pebble/results/step5_pendulum_chunks/angle_<angle>`

In [3]:
from pathlib import Path
import json
import os
import sys
import subprocess

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing src/ and notebooks/.')

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

TOTAL_STEPS = 200_000
EVAL_INTERVAL = 10_000
EVAL_EPISODES = 20
BUDGETS = [500, 1000, 3000]
DEVICE = 'cuda'
BASE_OUTPUT = Path('src/pebble/results/step5_pendulum_chunks')
SEED_CHUNKS = [(0, 4), (5, 9), (10, 14)]
ALL_ANGLES = [0, -60, 90, 120, -150]

print('Repo root:', REPO_ROOT)
print('Base output:', BASE_OUTPUT)
print('Seed chunks:', SEED_CHUNKS)
print('Python:', sys.executable)

Repo root: C:\Users\srksr\RL_projects\PA3\RL_PA3_Soft_Actor_critic
Base output: src\pebble\results\step5_pendulum_chunks
Seed chunks: [(0, 4), (5, 9), (10, 14)]
Python: c:\Users\srksr\miniconda3\python.exe


In [ ]:
from src.train.run_step5_pendulum_pebble import RunResult, _aggregate_method_results, _plot_target_comparison

def run_angle_seed_range(angle: int, seed_start: int, seed_end: int) -> None:
    out_dir = BASE_OUTPUT / f'angle_{angle}' / f'seeds_{seed_start}_{seed_end}'
    cmd = [
        sys.executable, '-m', 'src.train.run_step5_pendulum_pebble',
        '--device', DEVICE,
        '--angles', str(angle),
        '--seed-start', str(seed_start),
        '--seed-end', str(seed_end),
        '--total-steps', str(TOTAL_STEPS),
        '--eval-interval', str(EVAL_INTERVAL),
        '--eval-episodes', str(EVAL_EPISODES),
        '--budgets', *[str(x) for x in BUDGETS],
        '--output-dir', str(out_dir),
    ]

    print('Running angle:', angle, '| seeds:', seed_start, '-', seed_end)
    print('Output dir:', out_dir)
    print('Command:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    print('Completed angle:', angle, '| seeds:', seed_start, '-', seed_end)


def combine_angle_chunks(angle: int) -> Path:
    angle_dir = BASE_OUTPUT / f'angle_{angle}'
    chunk_dirs = [angle_dir / f'seeds_{s}_{e}' for s, e in SEED_CHUNKS]

    combined_runs: list[RunResult] = []
    chunk_summaries = []
    for chunk_dir in chunk_dirs:
        summary_path = chunk_dir / 'summary.json'
        if not summary_path.exists():
            raise FileNotFoundError(f'Missing chunk summary: {summary_path}')

        with summary_path.open('r', encoding='utf-8') as f:
            chunk_summary = json.load(f)
        chunk_summaries.append(chunk_summary)

        for run_dict in chunk_summary.get('runs', []):
            combined_runs.append(RunResult(**run_dict))

    target_runs: dict[str, list[RunResult]] = {'SAC (ground truth)': []}
    for budget in BUDGETS:
        target_runs[f'PEBBLE (budget={budget})'] = []

    for run in combined_runs:
        if run.method == 'sac_ground_truth':
            target_runs['SAC (ground truth)'].append(run)
        else:
            target_runs[f'PEBBLE (budget={run.preference_budget})'].append(run)

    aggregates = {label: _aggregate_method_results(runs) for label, runs in target_runs.items()}

    combined_dir = angle_dir / 'combined_0_14'
    combined_dir.mkdir(parents=True, exist_ok=True)

    _plot_target_comparison(
        angle,
        aggregates,
        combined_dir / 'plots' / f'target_{angle}_comparison.png',
    )

    combined_summary = {
        'angle': angle,
        'seed_chunks': [{'start': s, 'end': e} for s, e in SEED_CHUNKS],
        'chunk_paths': [str(path) for path in chunk_dirs],
        'chunk_summaries': chunk_summaries,
        'aggregates': aggregates,
        'runs': [run.__dict__ for run in combined_runs],
    }

    with (combined_dir / 'summary.json').open('w', encoding='utf-8') as f:
        json.dump(combined_summary, f, indent=2, ensure_ascii=True)

    print(f'Combined 15-seed summary saved: {combined_dir / "summary.json"}')
    return combined_dir / 'summary.json'

def combine_all_angles() -> Path:
    all_angle_summaries: dict[str, dict] = {}
    for angle in ALL_ANGLES:
        summary_path = combine_angle_chunks(angle)
        with summary_path.open('r', encoding='utf-8') as f:
            all_angle_summaries[str(angle)] = json.load(f)

    output_path = BASE_OUTPUT / 'combined_all_angles_summary.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(all_angle_summaries, f, indent=2, ensure_ascii=True)

    print(f'Combined all-angles summary saved: {output_path}')
    return output_path

## Run Angle 0

In [ ]:
# Change the range as needed: (0,4), (5,9), or (10,14)
run_angle_seed_range(0, 0, 4)

Running angle: 0 | seeds: 0 - 4
Output dir: src\pebble\results\step5_pendulum_chunks\angle_0\seeds_0_4
Command: c:\Users\srksr\miniconda3\python.exe -m src.train.run_step5_pendulum_pebble --device cuda --angles 0 --seed-start 0 --seed-end 4 --total-steps 200000 --eval-interval 10000 --eval-episodes 20 --budgets 500 1000 3000 --output-dir src\pebble\results\step5_pendulum_chunks\angle_0\seeds_0_4


In [ ]:
run_angle_seed_range(0, 5,9)

In [ ]:
run_angle_seed_range(0, 10, 14)

## Run Angle -60

In [ ]:
# Change the range as needed: (0,4), (5,9), or (10,14)
run_angle_seed_range(-60, 0, 4)

In [ ]:
run_angle_seed_range(-60, 5, 9)

In [ ]:
run_angle_seed_range(-60, 10, 14)

## Run Angle 90

In [ ]:
# Change the range as needed: (0,4), (5,9), or (10,14)
run_angle_seed_range(90, 0, 4)

In [ ]:
run_angle_seed_range(90, 5, 9)

In [ ]:
run_angle_seed_range(90, 10, 14)

## Run Angle 120

In [ ]:
# Change the range as needed: (0,4), (5,9), or (10,14)
run_angle_seed_range(120, 0, 4)

In [ ]:
run_angle_seed_range(120, 5, 9)

In [ ]:
run_angle_seed_range(120, 10, 14)

## Run Angle -150

In [ ]:
# Change the range as needed: (0,4), (5,9), or (10,14)
run_angle_seed_range(-150, 0, 4)

In [ ]:
run_angle_seed_range(-150, 5, 9)

In [ ]:
run_angle_seed_range(-150, 10, 14)

## Combine All Angles (Optional Final Summary)

In [ ]:
# Run this after all per-angle combine cells are done
combine_all_angles()

## Combine Chunk Logs Into 15-Seed Summary (Per Angle)

In [ ]:
# Run after all three chunks finish for angle 0
combine_angle_chunks(0)

# Run these after chunks complete for the other angles:
combine_angle_chunks(-60)
combine_angle_chunks(90)
combine_angle_chunks(120)
combine_angle_chunks(-150)

## Check Generated Folders

In [ ]:
for angle_dir in sorted(BASE_OUTPUT.glob('angle_*')):
    print('\n', angle_dir)
    for chunk_dir in sorted(angle_dir.glob('seeds_*')):
        print('  ', chunk_dir)